# 03 — Sequential vs Parallel Affine Scan

Compare sequential, Hillis-Steele, and chunked scan implementations.

Recurrence form: $h_t = a_t h_{t-1} + b_t$.

In [ ]:
import time
import torch
from mamba_minimal.parallel_scan import sequential_affine_scan, hillis_steele_affine_scan, chunked_affine_scan

torch.manual_seed(0)
a = torch.rand(4, 1024, 16)
b = torch.randn(4, 1024, 16)

ref = sequential_affine_scan(a, b)
par = hillis_steele_affine_scan(a, b)
chk = chunked_affine_scan(a, b, chunk_size=128)

print("max diff (parallel)", (ref - par).abs().max().item())
print("max diff (chunked)", (ref - chk).abs().max().item())

In [ ]:
# micro-timing on current device
for name, fn in [
    ("sequential", lambda: sequential_affine_scan(a, b)),
    ("parallel", lambda: hillis_steele_affine_scan(a, b)),
    ("chunked", lambda: chunked_affine_scan(a, b, chunk_size=128)),
]:
    start = time.perf_counter()
    _ = fn()
    dt_ms = (time.perf_counter() - start) * 1e3
    print(f"{name:>10}: {dt_ms:.2f} ms")